In [2]:
# Importation des packages et réglagle de la seed pour fixé un nombre de simulation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy_financial as npf

SEED = 2107
rng = np.random.default_rng(SEED)


In [3]:
# Paramètre de base pour les calcules

#Horizon temps
an_debut = 2027
an_fin = 2060
n_ans = an_fin - an_debut +1
cible = 3500000

#Revenue & dépense
salaire_base = 95000
Div_inc_base = 0
hypo_FIXE = 38900
fin_hypo = 2057
depense_base = 38000
fct_croissance_depense = 0.03
taux_depense_compressible = 0.12
taux_remb_dette = 0.20

#Bilan personnel au 31 Dec 2026
placement_base = 45000
propriete = 920000
inc_valeur_revente = 0
hypo_solde = 598000
autre_dette = 0

# Vérification des valeurs
valeur_nette_base = placement_base + propriete - hypo_solde - autre_dette
surplus_2026 = salaire_base - hypo_FIXE - depense_base - taux_depense_compressible * salaire_base

print("Nombre d'années:", n_ans)
print("Valeur nette:", valeur_nette_base)
print("Surplus 2026:", surplus_2026)

Nombre d'années: 34
Valeur nette: 367000
Surplus 2026: 6700.0


In [4]:
# Étape --- Hypothèque 

# Trouver le taux d'hypotèque
n_versements = fin_hypo - an_debut + 1

taux_hypo = npf.rate(nper=n_versements, pmt=-hypo_FIXE, pv=hypo_solde, fv=0)
print("taux Hypothèque :", round(taux_hypo,4))

# Faire le tableau d'amortissement
soldes = []
solde = hypo_solde
for annee in range(an_debut, an_fin + 1):
    if annee <= fin_hypo:
        interet = solde * taux_hypo
        solde = solde - (hypo_FIXE - interet)
    soldes.append(max(solde, 0))

soldes = np.array(soldes)
print("Solde en 2057 :", round(soldes[fin_hypo - an_debut],2))

taux Hypothèque : 0.0512
Solde en 2057 : 0.0


In [5]:
# Étape 3.1.1 --- Modélisation des Taux d'intérêt (bons du trésor)

taux_base = 0.0310
n_trimestre = n_ans *4

def simuler_taux(n_scenario, rng):
    taux = np.empty((n_scenario, n_trimestre))
    r = np.full(n_scenario, taux_base)

    for t in range(n_trimestre):
        z = rng.standard_normal(n_scenario)
        r = r + 0.1 * (0.03 - r) + 0.02 * np.sqrt(np.maximum(r, 0)) * z
        taux[:, t] = np.maximum(r, 0)

    return taux

# Vérification des résultats
taux_test = simuler_taux(2500, rng)
print(taux_test.shape)
print("taux min: ", round(taux_test.min(), 4))
print("Moyenne en 2060", round(taux_test[:, -1].mean(), 4))

### Entre pour chaque Trimestre 2.9% et 3.12% selon les simulations ###

(2500, 136)
taux min:  0.0054
Moyenne en 2060 0.0298


In [ ]:
# Étape 3.2 --- Modélisation des probabilité et du salaire

variation_salaire = np.array([0.05, 0.025, 0.015, -0.02])
prob_salaire = np.array([0.10, 0.40, 0.45, 0.05])

def simuler_salaire(n_scenario, rng):
    variation = rng.choice(variation_salaire, size=(n_scenario, n_ans), p=prob_salaire)
    return salaire_base * np.cumprod(1 + variation, axis=1)

# Vérification de la fonction
salaire_test = simuler_salaire(2500, rng)
print(salaire_test.shape)
print("Salaire moyen en 2027 : ", round(salaire_test[:, 0].mean()))
print("Salaire moyen en 2060 : ", round(salaire_test[:, -1].mean()))

(2500, 34)
Salaire moyen en 2027 :  97019
Salaire moyen en 2060 :  191546


In [7]:
# Importation du fichier Données_A26
Donnees = "Données_A26.xlsx"

prix_maison = pd.read_excel(Donnees, sheet_name="PrixMaison")
taux_hebdo  = pd.read_excel(Donnees, sheet_name="TauxInteret")
salaire_qc  = pd.read_excel(Donnees, sheet_name="Salaire")
population  = pd.read_excel(Donnees, sheet_name="Population")
sp500       = pd.read_excel(Donnees, sheet_name="SP500")
success     = pd.read_excel(Donnees, sheet_name="Success")

# Vérification 
print(prix_maison)
print(prix_maison.iloc[-1,1]) # devrait être 184.25

          Mois  Indice PrixMaison
0   1999-12-31             100.00
1   2000-01-31             100.59
2   2000-02-29             101.48
3   2000-03-31             101.14
4   2000-04-30             101.59
..         ...                ...
317 2026-05-31             179.60
318 2026-06-30             183.43
319 2026-07-31             183.84
320 2026-08-31             184.23
321 2026-09-30             184.25

[322 rows x 2 columns]
184.25


In [8]:
# Étape 3.1.2 --- Modéliser le salaire hebdomadaire Moyen

# Les données dans le fichier Donnees_A26 sont mensuelle donc resample au trimestre 
salaire_qc_trim = salaire_qc.set_index("Mois")["SalaireMoyen"].resample("QE").last()

#Distribution empirique des variation Trim
variations_salaire_qc = salaire_qc_trim.pct_change().dropna().to_numpy()

salaire_qc_Hebdo_base = 1275

def simuler_salaire_qc_hebdo(n_scenario, rng):
    variation = rng.choice(variations_salaire_qc, size=(n_scenario, n_trimestre))
    return salaire_qc_Hebdo_base * np.cumprod(1 + variation, axis=1)

# vérification Donnees
print("donnees Mens:", len(salaire_qc["SalaireMoyen"]))
print("donnes trim: " , len(salaire_qc_trim))
print("variation : ", len(variations_salaire_qc))

# Vérification Variation
print("----- Dist. Variation Empirique -----")
print("nb de variation", len(variations_salaire_qc))
print("Moyenne:", round(variations_salaire_qc.mean() * 100, 4), "%")
print("Écart-type :", round(variations_salaire_qc.std(ddof=1) * 100, 4), "%")
print("Min:", round(variations_salaire_qc.min() * 100, 2))
print("max:", round(variations_salaire_qc.max() * 100, 2))

# Vérification fonction
print("----- Fonction -----")
salaire_hebdo_test = simuler_salaire_qc_hebdo(2500, rng)
print(salaire_hebdo_test.shape)
print("Salaire 2060;", round(salaire_hebdo_test[:, -1].mean()), "$")

donnees Mens: 319
donnes trim:  107
variation :  106
----- Dist. Variation Empirique -----
nb de variation 106
Moyenne: 0.694 %
Écart-type : 0.9397 %
Min: -1.7
max: 5.61
----- Fonction -----
(2500, 136)
Salaire 2060; 3271 $


In [9]:
# Étape 3.1.3 --- Population
# Importation du PopAS_Qc_2026_sc_analyse
Population = "PopAS_Qc_2026_sc_analyse.xlsx"

# nettoyage et selection des colonnes
pop_proj = pd.read_excel(Population, sheet_name="Groupes d'âge", skiprows=7, header=None, usecols=[1, 3, 5, 6], names=['Scenario', 'Annee', 'Sexe', 'Population'])

# Selection référence (A2026) et (Sexe = 3).
pop_ref = pop_proj[(pop_proj['Scenario'] == "Référence (A2026)") & (pop_proj['Sexe'] == 3)]
pop_ref = pop_ref.set_index('Annee')['Population'].sort_index()

#Croissance 2027 à 2060
croissance_pop = pop_ref.pct_change().loc[2027 : 2060].to_numpy()

# Vérification des données
print("Nombre de croissances :", len(croissance_pop))
print("Population 2026:", int(pop_ref.loc[2026]))
print("Population 2060 :", int(pop_ref.loc[2060]))
print("Croissance moyenne:", round(croissance_pop.mean() * 100, 4), "%")
print("Min:", round(croissance_pop.min() * 100, 4), "%")
print("Max ;",round(croissance_pop.max() * 100, 4), "%")

Nombre de croissances : 34
Population 2026: 9027361
Population 2060 : 9174458
Croissance moyenne: 0.0476 %
Min: -0.1728 %
Max ; 0.2231 %


In [10]:
# Étape 3.1.4 --- Rendements du S&P 500

# Les données sont mensuelles
sp500_annuel = sp500.set_index("Mois")["SPX"].resample("YE").last()

# Rendements annuels observés de 1926 à 2025
rendements_sp500 = sp500_annuel.pct_change().dropna()

# Paramètres du modèle
sp500_mu    = rendements_sp500.mean()
sp500_sigma = rendements_sp500.std(ddof=1)


def simuler_sp500(n_scenario, rng):
    return sp500_mu + sp500_sigma * rng.standard_normal((n_scenario, n_ans))


# Vérification
print("Nombre de rendements annuels :", len(rendements_sp500))
print("Période:", rendements_sp500.index[0].year,"à", rendements_sp500.index[-1].year)
print("mu =", round(sp500_mu * 100, 4), "%")
print("sigma =", round(sp500_sigma * 100, 4), "%")

# Vérification de la fonctions pour simuler
sp500_test = simuler_sp500(2500, rng)
print("Dimensions :", sp500_test.shape)
print("Moyenne simulée :", round(sp500_test.mean() * 100, 4), "%")
print("Écart-type simulé :", round(sp500_test.std() * 100, 4), "%")

Nombre de rendements annuels : 100
Période: 1926 à 2025
mu = 8.3382 %
sigma = 18.9793 %
Dimensions : (2500, 34)
Moyenne simulée : 8.2997 %
Écart-type simulé : 18.9973 %


In [17]:
# Étape 3.1.5 --- Federal Fund Rate

def fedfund_variation(taux):
    niveau = taux[:, 3::4]
    return np.diff(niveau, axis=1, prepend=taux_base)


# Vérification
fedfund_test = fedfund_variation(taux_test)
print("Dimensions :", fedfund_test.shape)
print("Variation moyenne :", round(fedfund_test.mean(), 5)) # Très petit 0.0000X 
print("Écart-type:", round(fedfund_test.std(), 4))
print("Min :", round(fedfund_test.min(), 4))
print("Max : ", round(fedfund_test.max(), 4))

Dimensions : (2500, 34)
Variation moyenne : -3e-05
Écart-type: 0.0066
Min : -0.0362
Max :  0.0288


In [20]:
# Étape 3.3.1 --- Régression logistique des probabilités d'une année de succès
import statsmodels.api as sm

# X1 = rendement annuel du S&P 500, X2 = variation annuelle du Fed Funds
X = sm.add_constant(success[["SPX", "FedFundRate"]])
y = success["Success"]

modele_logit = sm.Logit(y, X).fit()
print(modele_logit.summary())

logit_b0 = modele_logit.params["const"]
logit_b1 = modele_logit.params["SPX"]
logit_b2 = modele_logit.params["FedFundRate"]


def proba_succes(rendement_sp500, variation_fedfund):
    z = logit_b0 + logit_b1 * rendement_sp500 + logit_b2 * variation_fedfund
    return 1 / (1 + np.exp(-z))


# Vérification des coefficients
print("--- params ---")
print("b0 =", round(logit_b0, 4))
print("b1 =", round(logit_b1, 4))
print("b2 =", round(logit_b2, 4))

# Vérification
proba_test = proba_succes(sp500_test, fedfund_test)
print("Dimensions :", proba_test.shape)        # (2500, 34)
print("P moyenne  :", round(proba_test.mean(), 4))

Optimization terminated successfully.
         Current function value: 0.272822
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                Success   No. Observations:                   65
Model:                          Logit   Df Residuals:                       62
Method:                           MLE   Df Model:                            2
Date:                Sun, 15 Nov 2026   Pseudo R-squ.:                  0.5905
Time:                        13:18:33   Log-Likelihood:                -17.733
converged:                       True   LL-Null:                       -43.308
Covariance Type:            nonrobust   LLR p-value:                 7.818e-12
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -1.3804      0.660     -2.091      0.036      -2.674      -0.087
SPX            24.7886    

In [35]:
# Étape 3.3.2 --- Scénario selon les probabilitées

def classer_scenario(proba):
    return np.where(proba > 0.75, 0, np.where(proba < 0.25, 2, 1))

# Vérification sur les probabilités simulées
scenario_test = classer_scenario(proba_test)
for code, nom in enumerate(["important", "modéré", "échec"]):
    print(f"{nom} : {(scenario_test == code).mean() * 100:.1f} %")

important : 46.4 %
modéré : 18.3 %
échec : 35.3 %


In [28]:
# Étape 3.3.3 --- Probabilité conditionnelle des scénarios

elevee = np.array([0.60, 0.20, 0.20])
faible = np.array([0.10, 0.35, 0.55])

croyance_priori = 0.75
seuil_faible = 0.80

def croyance(p_elevee, scenario):
    num = elevee[scenario] * p_elevee
    deno = num + faible[scenario] * (1-p_elevee)
    return num / deno

# Vérification de la fonction
p = croyance_priori
for annee in range(1,5):
    p = croyance(p,2)
    print(f"Après {annee} échec(s) : P(élevée) = {p:.4f} P(faible) = {1-p:.4f}")

Après 1 échec(s) : P(élevée) = 0.5217 P(faible) = 0.4783
Après 2 échec(s) : P(élevée) = 0.2840 P(faible) = 0.7160
Après 3 échec(s) : P(élevée) = 0.1261 P(faible) = 0.8739
Après 4 échec(s) : P(élevée) = 0.0498 P(faible) = 0.9502


In [38]:
# Étape 3.3.4 --- conséquence selon les scénarios

ent_valeur_base = 100000
ent_seuil_fermeture = 50000
salaire_salariee = 95000

# par scénario 
ent_croissance = np.array([0.25, 0.10, -0.15])
ent_salaire = np.array([150000, 120000, 90000])
ent_dividende = np.array([0.05, 0.02, 0])

def simuler_entreprise(scenario):
    n = scenario.shape[0]
    salaire = np.zeros((n, n_ans))
    dividende = np.zeros((n, n_ans))
    valeur = np.full(n, float(ent_valeur_base))
    croyance_p = np.full(n , croyance_priori)
    ouverte = np.ones(n, dtype=bool)

    for t in range(n_ans):
        s = scenario[:, t]
        valeur = np.where(ouverte, valeur * (1 + ent_croissance[s]), valeur)
        salaire[:, t] = np.where(ouverte, ent_salaire[s], salaire_salariee)
        dividende[:, t] = np.where(ouverte, ent_dividende[s] * valeur, 0.0)
        croyance_p = np.where(ouverte, croyance(croyance_p, s), croyance_p)
        ferme = ouverte & ((valeur < ent_seuil_fermeture) | ((1 - croyance_p) > seuil_faible))
        dividende[:, t] += np.where(ferme, valeur, 0.0)
        valeur = np.where(ferme, 0.0, valeur)
        ouverte = ouverte & ~ferme
    return salaire, dividende



# Vérification
sal_ent, div_ent = simuler_entreprise(scenario_test)
print("Salaire moyen 2027 :", round(sal_ent[:, 0].mean()))
print("Salaire moyen 2060 :", round(sal_ent[:, -1].mean()))
print("Dividendes cumulés :", round(div_ent.sum(axis=1).mean()))

Salaire moyen 2027 : 123156
Salaire moyen 2060 : 115664
Dividendes cumulés : 488774
